# STARLINK-32838四个VHF时间窗口的几何与理想阵列增益

本Notebook把21CMA E08站心、两次实测137.055 MHz事件、两次地平线上未检出预测窗口、SpaceX-E SupGP轨道和127阵元几何放入同一条可复核流程，计算相对NCP波束中心的理想阵列因子。



## 本Notebook的计算目标

本Notebook回答：

1. 四个窗口内卫星的方位角、高度角和NCP夹角范围；
2. 在137.055 MHz下由127阵元几何决定的理想归一化功率响应；
3. 在约1.028 s完整窗口内的线性功率时间平均增益；
4. 理想阵列因子作为流量密度初步旁瓣修正时的适用边界。

`EW_position.csv`的E08六个测点只用于确定站心；`ant_pos.py`提供127个唯一阵元坐标用于理想阵列因子。

## 1. 路径和固定参数

- 射频中心：137.055 MHz；
- 观测到的LoRa信号带宽约41.667 kHz；对如此窄的相对带宽，先在中心频率计算beam通常足够；
- 波束中心按21CMA本次观测的NCP指向处理：赤纬+90°，在E08本地对应方位角0°、高度角等于台站纬度；
- 事件时间使用acquisition metadata重建的接收时间；SupGP直接传播到该UTC，不加约+0.020 s状态匹配偏移。该偏移只作为可选敏感性检查参数保留。

In [ ]:
from pathlib import Path
from datetime import datetime, timedelta, timezone
import csv
import json
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sgp4 import omm
from sgp4.api import Satrec
from skyfield.api import EarthSatellite, load, wgs84

def find_project_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'project_config.py').is_file():
            return candidate
    raise FileNotFoundError('Run from this repository or one of its subdirectories')

PROJECT_ROOT = find_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from project_config import DATA_DIR, RAW_DATA_DIR, RESULTS_DIR, file_start_utc
PROJECT_DIR = PROJECT_ROOT
FLUX_DIR = PROJECT_ROOT / '5_flux_density'
OUTPUT_DIR = RESULTS_DIR / '5_flux_density' / 'oskar_input'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COORDINATE_FILE = DATA_DIR / 'station' / 'EW_position.csv'
RAW_FILES = {
    'Event 1': RAW_DATA_DIR / '20250415-2200-0.dat',
    'Event 2': RAW_DATA_DIR / '20250415-2200-12.dat',
}
SUPGP_FILE = DATA_DIR / 'orbital' / 'STARLINK-32838_SpaceX-E_SupGP.csv'
SUPGP_SUMMARY_FILE = RESULTS_DIR / '3_satellite_identification' / 'supgp_validation_summary.json'
FRAME_FILES = {
    'Event 1': RESULTS_DIR / '2_demodulation' / 'file0' / 'frames.csv',
    'Event 2': RESULTS_DIR / '2_demodulation' / 'file12' / 'frames.csv',
}

RAW_SAMPLE_RATE_HZ = 480_000_000.0
RAW_DTYPE = np.dtype('<i2')
RF_CENTER_HZ = 137_055_000.0
OBSERVED_SIGNAL_BANDWIDTH_HZ = 41_666.6666666667
SPEED_OF_LIGHT_M_S = 299_792_458.0
WAVELENGTH_M = SPEED_OF_LIGHT_M_S / RF_CENTER_HZ
NCP_RA_J2000_DEG = 0.0  # 在北天极，RA数值本身不影响指向。
NCP_DEC_J2000_DEG = 90.0

print('Output directory:', OUTPUT_DIR)
print(f'RF center: {RF_CENTER_HZ / 1e6:.6f} MHz')
print(f'Wavelength: {WAVELENGTH_M:.9f} m')


## 2. 重建四个事件的接收或预测时间范围

Event 1和Event 2是实际解调出的包：将提交的原始`.dat`文件采集起始UTC分别加上LFdemod给出的`preamble_file_s`和`stop_file_s`，重建包的绝对UTC区间。

Missing 1和Missing 2没有实际检出信号，不能称为实测事件。它们的名义起点由两个实测preamble的acquisition metadata重建UTC间隔向后外推，名义时长取两个实测包的共同时长。四个时间窗口都保留完整微秒精度，不加SupGP的+0.019952 s状态匹配偏移。



In [ ]:
def raw_file_duration_seconds(path):
    sample_count = path.stat().st_size // RAW_DTYPE.itemsize
    return sample_count / RAW_SAMPLE_RATE_HZ

def raw_file_number(path):
    return int(path.stem.rsplit('-', 1)[1])

observed_rows = []
for cycle_index, event_id in enumerate(('Event 1', 'Event 2')):
    raw_path = RAW_FILES[event_id]
    frame_path = FRAME_FILES[event_id]
    if not raw_path.is_file() or not frame_path.is_file():
        raise FileNotFoundError(raw_path if not raw_path.is_file() else frame_path)
    frame_table = pd.read_csv(frame_path)
    if len(frame_table) != 1:
        raise ValueError(f'Expected exactly one decoded frame in {frame_path}')
    frame = frame_table.iloc[0]
    file_start = file_start_utc(raw_path.name)
    packet_start_utc = file_start + pd.to_timedelta(float(frame['preamble_file_s']), unit='s')
    packet_stop_utc = file_start + pd.to_timedelta(float(frame['stop_file_s']), unit='s')
    packet_duration_s = float(frame['stop_file_s'] - frame['preamble_file_s'])
    observed_rows.append({
        'event_id': event_id, 'cycle_index_from_event1': cycle_index,
        'event_status': 'observed_and_decoded',
        'preamble_start_utc': packet_start_utc,
        'packet_midpoint_utc': packet_start_utc + pd.to_timedelta(packet_duration_s / 2, unit='s'),
        'packet_stop_utc': packet_stop_utc, 'packet_duration_s': packet_duration_s,
        'source_file': str(raw_path), 'source_file_number': raw_file_number(raw_path),
        'preamble_start_file_s': float(frame['preamble_file_s']),
        'packet_stop_file_s': float(frame['stop_file_s']),
        'carrier_offset_hz': float(frame['carrier_offset_hz']),
        'time_basis': 'committed file-start UTC plus LFdemod sample time',
        'supgp_time_correction_applied_s': 0.0,
    })

observed_intervals = pd.DataFrame(observed_rows).sort_values('cycle_index_from_event1').reset_index(drop=True)
observed_cadence_s = (observed_intervals.loc[1, 'preamble_start_utc'] - observed_intervals.loc[0, 'preamble_start_utc']).value / 1e9
nominal_packet_duration_s = float(observed_intervals['packet_duration_s'].mean())
predicted_rows = []
event1_start_utc = observed_intervals.loc[0, 'preamble_start_utc']
for cycle_index, event_id in ((2, 'Missing 1'), (3, 'Missing 2')):
    packet_start_utc = event1_start_utc + pd.to_timedelta(cycle_index * observed_cadence_s, unit='s')
    packet_stop_utc = packet_start_utc + pd.to_timedelta(nominal_packet_duration_s, unit='s')
    predicted_rows.append({
        'event_id': event_id, 'cycle_index_from_event1': cycle_index,
        'event_status': 'cadence_predicted_not_detected',
        'preamble_start_utc': packet_start_utc,
        'packet_midpoint_utc': packet_start_utc + pd.to_timedelta(nominal_packet_duration_s / 2, unit='s'),
        'packet_stop_utc': packet_stop_utc, 'packet_duration_s': nominal_packet_duration_s,
        'source_file': '', 'source_file_number': pd.NA,
        'preamble_start_file_s': np.nan, 'packet_stop_file_s': np.nan,
        'carrier_offset_hz': np.nan,
        'time_basis': 'Event 1 UTC plus cycle index times measured cadence',
        'supgp_time_correction_applied_s': 0.0,
    })

event_intervals = pd.concat([observed_intervals, pd.DataFrame(predicted_rows)], ignore_index=True).sort_values('cycle_index_from_event1').reset_index(drop=True)
event_intervals['rf_center_hz'] = RF_CENTER_HZ
event_intervals['observed_bandwidth_hz'] = OBSERVED_SIGNAL_BANDWIDTH_HZ
time_output_path = OUTPUT_DIR / 'four_event_time_windows.csv'
event_intervals.to_csv(time_output_path, index=False, float_format='%.9f')
print(f'Acquisition-metadata-derived cadence: {observed_cadence_s:.9f} s')
display(event_intervals)


## 3. 计算四个时间窗口内的卫星方向范围

本节用E08六个ECEF测点的均值定义站心，并将CelesTrak `SpaceX-E` SupGP直接传播到第2节的acquisition metadata接收或预测UTC，**不加+0.019952 s偏移**。

每个约1.027584 s窗口按目标5 ms间隔均匀取样，并强制包含起点和终点。保存每个样本的：

- 从正北向正东增加的方位角；
- 相对WGS84几何地平的高度角；
- 卫星方向与NCP固定波束中心的球面夹角；
- 斜距。

角度范围表是后续beam计算的几何输入；Missing 1/2仍只是周期外推的名义窗口。

In [ ]:
import importlib.util

GEOMETRY_TARGET_STEP_S = 0.005


def ecef_to_wgs84(x_m, y_m, z_m):
    """Convert ECEF metres to WGS84 longitude, latitude and ellipsoidal height."""
    semi_major_axis_m = 6_378_137.0
    flattening = 1.0 / 298.257223563
    eccentricity_sq = flattening * (2.0 - flattening)
    longitude_rad = np.arctan2(y_m, x_m)
    horizontal_m = np.hypot(x_m, y_m)
    latitude_rad = np.arctan2(z_m, horizontal_m * (1.0 - eccentricity_sq))
    for _ in range(20):
        prime_vertical_m = semi_major_axis_m / np.sqrt(
            1.0 - eccentricity_sq * np.sin(latitude_rad) ** 2
        )
        height_m = horizontal_m / np.cos(latitude_rad) - prime_vertical_m
        updated_latitude_rad = np.arctan2(
            z_m,
            horizontal_m * (
                1.0 - eccentricity_sq * prime_vertical_m
                / (prime_vertical_m + height_m)
            ),
        )
        if abs(updated_latitude_rad - latitude_rad) < 1e-15:
            latitude_rad = updated_latitude_rad
            break
        latitude_rad = updated_latitude_rad
    prime_vertical_m = semi_major_axis_m / np.sqrt(
        1.0 - eccentricity_sq * np.sin(latitude_rad) ** 2
    )
    height_m = horizontal_m / np.cos(latitude_rad) - prime_vertical_m
    return np.degrees(longitude_rad), np.degrees(latitude_rad), height_m


coordinate_columns = [
    'point', 'x_ecef_m', 'y_ecef_m', 'z_ecef_m',
    'latitude_ddmm', 'longitude_ddmm', 'survey_height_m',
]
coordinate_table = pd.read_csv(
    COORDINATE_FILE, sep=r'\s+', header=None, names=coordinate_columns
)
e08_points = coordinate_table[
    coordinate_table['point'].str.lower().str.fullmatch(r'e08\.[1-6]')
].copy()
if len(e08_points) != 6:
    raise ValueError(f'Expected six E08 survey points; found {len(e08_points)}')
e08_ecef_m = e08_points[['x_ecef_m', 'y_ecef_m', 'z_ecef_m']].mean().to_numpy(float)
site_longitude_deg, site_latitude_deg, site_height_m = ecef_to_wgs84(*e08_ecef_m)
station_center = pd.DataFrame([{
    'station': 'E08',
    'survey_point_count': 6,
    'x_ecef_m': e08_ecef_m[0],
    'y_ecef_m': e08_ecef_m[1],
    'z_ecef_m': e08_ecef_m[2],
    'longitude_deg_east': site_longitude_deg,
    'latitude_deg_north': site_latitude_deg,
    'ellipsoidal_height_m': site_height_m,
}])
station_center.to_csv(OUTPUT_DIR / 'e08_station_center.csv', index=False, float_format='%.12f')

with SUPGP_FILE.open(newline='') as handle:
    raw_supgp_rows = list(csv.DictReader(handle))
if not raw_supgp_rows:
    raise ValueError(f'No SupGP rows in {SUPGP_FILE}')
if {row['NORAD_CAT_ID'] for row in raw_supgp_rows} != {'62827'}:
    raise ValueError('SupGP file must contain only NORAD 62827')
supgp_entries = sorted([
    {
        'raw': row,
        'epoch_utc': pd.Timestamp(row['EPOCH'], tz='UTC'),
    }
    for row in raw_supgp_rows
], key=lambda item: item['epoch_utc'])
timescale = load.timescale()
observer = wgs84.latlon(
    latitude_degrees=site_latitude_deg,
    longitude_degrees=site_longitude_deg,
    elevation_m=site_height_m,
)


def nearest_supgp_entry(utc):
    return min(
        supgp_entries,
        key=lambda entry: abs((entry['epoch_utc'] - pd.Timestamp(utc)).value),
    )


def satellite_from_supgp(entry):
    satrec = Satrec()
    omm.initialize(satrec, entry['raw'])
    return EarthSatellite.from_satrec(satrec, timescale)


site_latitude_rad = np.deg2rad(site_latitude_deg)
ncp_enu = np.array([0.0, np.cos(site_latitude_rad), np.sin(site_latitude_rad)])
direction_rows = []
direction_range_rows = []
for _, event in event_intervals.iterrows():
    event_id = str(event['event_id'])
    event_status = str(event['event_status'])
    start_utc = pd.Timestamp(event['preamble_start_utc'])
    duration_s = float(event['packet_duration_s'])
    interval_count = int(np.ceil(duration_s / GEOMETRY_TARGET_STEP_S))
    offsets_s = np.linspace(0.0, duration_s, interval_count + 1)
    sample_timestamps = pd.DatetimeIndex([
        start_utc + pd.to_timedelta(offset_s, unit='s')
        for offset_s in offsets_s
    ])
    midpoint_utc = start_utc + pd.to_timedelta(duration_s / 2.0, unit='s')
    entry = nearest_supgp_entry(midpoint_utc)
    satellite = satellite_from_supgp(entry)
    fractional_second = (
        sample_timestamps.second.to_numpy(float)
        + sample_timestamps.microsecond.to_numpy(float) / 1e6
        + sample_timestamps.nanosecond.to_numpy(float) / 1e9
    )
    skyfield_times = timescale.utc(
        sample_timestamps.year.to_numpy(int),
        sample_timestamps.month.to_numpy(int),
        sample_timestamps.day.to_numpy(int),
        sample_timestamps.hour.to_numpy(int),
        sample_timestamps.minute.to_numpy(int),
        fractional_second,
    )
    topocentric = (satellite - observer).at(skyfield_times)
    altitude, azimuth, distance = topocentric.altaz()
    azimuth_deg = np.asarray(azimuth.degrees, dtype=float)
    elevation_deg = np.asarray(altitude.degrees, dtype=float)
    range_km = np.asarray(distance.km, dtype=float)
    azimuth_rad = np.deg2rad(azimuth_deg)
    elevation_rad = np.deg2rad(elevation_deg)
    source_enu = np.column_stack([
        np.cos(elevation_rad) * np.sin(azimuth_rad),
        np.cos(elevation_rad) * np.cos(azimuth_rad),
        np.sin(elevation_rad),
    ])
    ncp_separation_deg = np.rad2deg(np.arccos(np.clip(source_enu @ ncp_enu, -1.0, 1.0)))
    unwrapped_azimuth_deg = np.rad2deg(np.unwrap(azimuth_rad))
    step_separations_deg = np.rad2deg(np.arccos(np.clip(
        np.sum(source_enu[:-1] * source_enu[1:], axis=1), -1.0, 1.0
    )))
    for index, offset_s in enumerate(offsets_s):
        direction_rows.append({
            'event_id': event_id,
            'event_status': event_status,
            'sample_index': index,
            'offset_from_window_start_s': float(offset_s),
            'sample_utc': sample_timestamps[index],
            'azimuth_deg_north_through_east': azimuth_deg[index],
            'elevation_deg': elevation_deg[index],
            'ncp_separation_deg': ncp_separation_deg[index],
            'slant_range_km': range_km[index],
            'enu_east': source_enu[index, 0],
            'enu_north': source_enu[index, 1],
            'enu_up': source_enu[index, 2],
            'supgp_epoch_utc': entry['epoch_utc'],
            'supgp_fit_rms_km': float(entry['raw']['RMS']),
            'supgp_time_correction_applied_s': 0.0,
        })
    direction_range_rows.append({
        'event_id': event_id,
        'event_status': event_status,
        'window_start_utc': start_utc,
        'window_stop_utc': start_utc + pd.to_timedelta(duration_s, unit='s'),
        'duration_s': duration_s,
        'sample_count': len(offsets_s),
        'actual_sample_step_s': float(offsets_s[1] - offsets_s[0]),
        'azimuth_start_deg': azimuth_deg[0],
        'azimuth_stop_deg': azimuth_deg[-1],
        'azimuth_span_deg': float(np.ptp(unwrapped_azimuth_deg)),
        'elevation_start_deg': elevation_deg[0],
        'elevation_stop_deg': elevation_deg[-1],
        'elevation_min_deg': float(np.min(elevation_deg)),
        'elevation_max_deg': float(np.max(elevation_deg)),
        'ncp_separation_start_deg': ncp_separation_deg[0],
        'ncp_separation_stop_deg': ncp_separation_deg[-1],
        'ncp_separation_min_deg': float(np.min(ncp_separation_deg)),
        'ncp_separation_max_deg': float(np.max(ncp_separation_deg)),
        'angular_track_length_deg': float(np.sum(step_separations_deg)),
        'slant_range_start_km': range_km[0],
        'slant_range_stop_km': range_km[-1],
    })

direction_timeseries = pd.DataFrame(direction_rows)
direction_ranges = pd.DataFrame(direction_range_rows)
direction_timeseries.to_csv(
    OUTPUT_DIR / 'four_event_direction_timeseries.csv', index=False, float_format='%.12f'
)
direction_ranges.to_csv(
    OUTPUT_DIR / 'four_event_direction_ranges.csv', index=False, float_format='%.12f'
)
print(f'E08 WGS84: lon={site_longitude_deg:.9f} deg, lat={site_latitude_deg:.9f} deg, h={site_height_m:.3f} m')
display(direction_ranges)


## 4. 127阵元理想阵列因子和全包平均增益


以NCP为理想波束中心，等权阵列的归一化电压阵列因子为

$$A(t)=\frac{1}{N}\sum_{k=1}^{N}\exp\left[i\frac{2\pi}{\lambda}\,\mathbf r_k\cdot(\hat{\mathbf s}(t)-\hat{\mathbf s}_0)\right],$$

归一化功率响应为$G(t)=|A(t)|^2$，NCP波束中心的$G=1$。对每个完整时间窗口在**线性功率域**做梯形积分：

$$\bar G=\frac{1}{T}\int_0^T G(t)\,dt,\qquad G_{\rm dB}=10\log_{10}\bar G.$$

这个结果是只由阵元几何决定的**相对NCP理想功率增益**，不是经实测标定的绝对增益。它忽略单阵元方向图、极化、互耦、地面、模拟延迟量化、坏阵元与幅相误差。

In [ ]:
ANTENNA_POSITION_FILE = PROJECT_ROOT / '5_flux_density' / 'ant_pos.py'

spec = importlib.util.spec_from_file_location('e08_ant_pos', ANTENNA_POSITION_FILE)
antenna_module = importlib.util.module_from_spec(spec)
if spec.loader is None:
    raise ImportError(f'Cannot load {ANTENNA_POSITION_FILE}')
spec.loader.exec_module(antenna_module)
antenna_rows = np.asarray(antenna_module.data, dtype=float)
antenna_enu_m, coordinate_counts = np.unique(
    antenna_rows, axis=0, return_counts=True
)
duplicate_rows = antenna_enu_m[coordinate_counts > 1]
if antenna_rows.shape != (128, 3):
    raise ValueError(f'Expected ant_pos.py shape (128, 3); got {antenna_rows.shape}')
if antenna_enu_m.shape != (127, 3):
    raise ValueError(f'Expected 127 unique antenna positions; got {antenna_enu_m.shape[0]}')
if duplicate_rows.shape != (1, 3) or not np.allclose(duplicate_rows[0], 0.0):
    raise ValueError(f'Unexpected duplicate antenna rows: {duplicate_rows}')
antenna_enu_m = antenna_enu_m - antenna_enu_m.mean(axis=0, keepdims=True)

wave_number_rad_m = 2.0 * np.pi / WAVELENGTH_M
boresight_phase_rad = wave_number_rad_m * ((ncp_enu - ncp_enu) @ antenna_enu_m.T)
boresight_power_gain = np.abs(np.exp(1j * boresight_phase_rad).mean()) ** 2
if not np.isclose(boresight_power_gain, 1.0, rtol=0.0, atol=1e-14):
    raise AssertionError(f'NCP normalization failed: {boresight_power_gain}')
source_enu = direction_timeseries[['enu_east', 'enu_north', 'enu_up']].to_numpy(float)
direction_delta = source_enu - ncp_enu[None, :]
phase_rad = wave_number_rad_m * (direction_delta @ antenna_enu_m.T)
voltage_array_factor = np.exp(1j * phase_rad).mean(axis=1)
power_gain_linear = np.abs(voltage_array_factor) ** 2
power_gain_db = 10.0 * np.log10(
    np.maximum(power_gain_linear, np.finfo(float).tiny)
)
array_factor_timeseries = direction_timeseries.copy()
array_factor_timeseries['ideal_voltage_array_factor_abs'] = np.abs(voltage_array_factor)
array_factor_timeseries['ideal_power_gain_relative_ncp_linear'] = power_gain_linear
array_factor_timeseries['ideal_power_gain_relative_ncp_db'] = power_gain_db

gain_summary_rows = []
for _, event in event_intervals.iterrows():
    event_id = str(event['event_id'])
    rows = array_factor_timeseries[
        array_factor_timeseries['event_id'] == event_id
    ].sort_values('offset_from_window_start_s')
    offsets_s = rows['offset_from_window_start_s'].to_numpy(float)
    gains_linear = rows['ideal_power_gain_relative_ncp_linear'].to_numpy(float)
    average_gain_linear = float(
        np.trapz(gains_linear, offsets_s) / float(event['packet_duration_s'])
    )
    midpoint_index = int(np.argmin(np.abs(
        offsets_s - float(event['packet_duration_s']) / 2.0
    )))
    direction_row = direction_ranges[direction_ranges['event_id'] == event_id].iloc[0]
    gain_summary_rows.append({
        'event_id': event_id,
        'event_status': event['event_status'],
        'window_start_utc': event['preamble_start_utc'],
        'window_stop_utc': event['packet_stop_utc'],
        'duration_s': float(event['packet_duration_s']),
        'time_sample_count': len(rows),
        'frequency_hz': RF_CENTER_HZ,
        'antenna_count': len(antenna_enu_m),
        'coordinate_convention_assumed': 'ant_pos columns = local East, North, Up metres',
        'beam_center': 'NCP',
        'azimuth_start_deg': direction_row['azimuth_start_deg'],
        'azimuth_stop_deg': direction_row['azimuth_stop_deg'],
        'elevation_min_deg': direction_row['elevation_min_deg'],
        'elevation_max_deg': direction_row['elevation_max_deg'],
        'ncp_separation_min_deg': direction_row['ncp_separation_min_deg'],
        'ncp_separation_max_deg': direction_row['ncp_separation_max_deg'],
        'angular_track_length_deg': direction_row['angular_track_length_deg'],
        'midpoint_power_gain_linear': gains_linear[midpoint_index],
        'midpoint_power_gain_db': 10.0 * np.log10(max(gains_linear[midpoint_index], np.finfo(float).tiny)),
        'time_averaged_power_gain_linear': average_gain_linear,
        'time_averaged_power_gain_db': 10.0 * np.log10(max(average_gain_linear, np.finfo(float).tiny)),
        'instantaneous_gain_min_linear': float(np.min(gains_linear)),
        'instantaneous_gain_max_linear': float(np.max(gains_linear)),
        'instantaneous_gain_min_db': float(np.min(rows['ideal_power_gain_relative_ncp_db'])),
        'instantaneous_gain_max_db': float(np.max(rows['ideal_power_gain_relative_ncp_db'])),
        'supgp_time_correction_applied_s': 0.0,
    })

ideal_gain_summary = pd.DataFrame(gain_summary_rows)
array_factor_timeseries.to_csv(
    OUTPUT_DIR / 'four_event_ideal_array_factor_timeseries.csv',
    index=False, float_format='%.12e',
)
ideal_gain_summary.to_csv(
    OUTPUT_DIR / 'four_event_ideal_array_gain_summary.csv',
    index=False, float_format='%.12e',
)

print('Input antenna rows:', len(antenna_rows))
print('Unique antenna positions used:', len(antenna_enu_m))
print('Removed duplicate coordinate:', duplicate_rows.tolist())
display(ideal_gain_summary)


## 5. 事件1和事件2的主瓣等效表观流量密度

本节只从原始电压数据的信号与背景功率差出发，计算两个已解调事件的 **nominal-main-beam-equivalent apparent flux density（主瓣等效表观流量密度）**。

### 5.1 数据量与计算关系

从原始int16电压数据重新DDC到137.055 MHz，用240 kS/s复IQ计算线性PSD，然后在41.6667 kHz LoRa信道内积分。这与瀑布图使用同类时频分析，但保留了线性功率，避免受图像色标和dB显示范围影响。

- `on`：从LFdemod给出的preamble start到packet stop；
- `off`：事件前各取1.0 s和事件后各取1.0 s，与事件边界保留0.30 s guard；
- STFT：Hann窗，4096点，不重叠；每个功率样本覆盖17.067 ms，bin spacing为58.594 Hz；
- 功率超额比：$R=\langle P_{\rm on}\rangle/\langle P_{\rm off}\rangle-1$；
- 采用的137.055 MHz主瓣SEFD：$\mathrm{SEFD}=2k_{\rm B}T_{\rm sys}/A_{\rm eff}$；
- 表观流量密度：$S_{\rm app}=R\,\mathrm{SEFD}$。

采用前期单pod灵敏度估计中的 $A_{\rm eff}=218\,\mathrm{m^2}$、150 MHz处 $T_{\rm sky}=253.2$ K和 $T_{\rm sys}=350$ K，并以$T_{\rm sky}\propto\nu^{-2.55}$缩放到137.055 MHz。因为on/off使用同一信道带宽，链路增益和带宽在功率比中抵消；带宽已通过PSD积分进入$P_{\rm on}$与$P_{\rm off}$。



In [ ]:
import importlib.util
import sys
from IPython.display import Image

APPARENT_FLUX_SCRIPT = FLUX_DIR / 'calculate_apparent_flux.py'
APPARENT_FLUX_OUTPUT_DIR = RESULTS_DIR / '5_flux_density' / 'apparent_flux'

spec = importlib.util.spec_from_file_location('starlink_apparent_flux_analysis', APPARENT_FLUX_SCRIPT)
if spec is None or spec.loader is None:
    raise RuntimeError(f'Cannot load {APPARENT_FLUX_SCRIPT}')
apparent_flux_module = importlib.util.module_from_spec(spec)
sys.modules[spec.name] = apparent_flux_module
spec.loader.exec_module(apparent_flux_module)
apparent_flux_module.OUTPUT_DIR = APPARENT_FLUX_OUTPUT_DIR

events = [apparent_flux_module.load_event(path) for path in apparent_flux_module.SUMMARY_PATHS]
event_name = {'file0': 'Event 1', 'file12': 'Event 2'}
apparent_flux_results = []
apparent_flux_series = []

for event in events:
    print(f'Processing {event_name.get(event.label, event.label)}: {event.input_path}')
    result, times, powers = apparent_flux_module.measure_event(event)
    result['event_id'] = event_name.get(event.label, event.label)
    result['nominal_main_beam_apparent_flux_density_kjy'] = (
        result['nominal_main_beam_apparent_flux_density_jy'] / 1e3
    )
    result['nominal_main_beam_apparent_flux_density_formal_sem_kjy'] = (
        result['nominal_main_beam_apparent_flux_density_formal_sem_jy'] / 1e3
    )
    apparent_flux_results.append(result)
    apparent_flux_series.append((event, times, powers))

apparent_flux_module.write_outputs(apparent_flux_results, apparent_flux_series)
apparent_flux_summary = pd.DataFrame(apparent_flux_results)

calibration = apparent_flux_module.calibration_values()
print('Adopted T_sky at 137.055 MHz (K):', calibration['sky_temperature_137mhz_k'])
print('Adopted T_sys at 137.055 MHz (K):', calibration['system_temperature_137mhz_k'])
print('Nominal main-beam SEFD at 137.055 MHz (Jy):', calibration['nominal_main_beam_sefd_137mhz_jy'])
print('Event 1 / Event 2 apparent-flux ratio:', (
    apparent_flux_summary.loc[apparent_flux_summary['event_id'] == 'Event 1',
                                'nominal_main_beam_apparent_flux_density_jy'].iloc[0]
    / apparent_flux_summary.loc[apparent_flux_summary['event_id'] == 'Event 2',
                                  'nominal_main_beam_apparent_flux_density_jy'].iloc[0]
))

display_columns = [
    'event_id', 'preamble_start_s', 'packet_stop_s', 'carrier_offset_hz',
    'on_spectrum_count', 'off_spectrum_count', 'power_ratio_on_off',
    'excess_power_ratio',
    'nominal_main_beam_apparent_flux_density_kjy',
    'nominal_main_beam_apparent_flux_density_formal_sem_kjy',
    'adc_clip_fraction', 'adc_near_full_scale_fraction',
]
display(apparent_flux_summary[display_columns])
display(Image(filename=str(APPARENT_FLUX_OUTPUT_DIR / 'channel_power_timeseries.png')))
print('Saved outputs to:', APPARENT_FLUX_OUTPUT_DIR)


## 6. Missing 1和Missing 2相对Event 2的路径衰减

仅计算自由空间传播中 $P\propto R^{-2}$ 带来的相对衰减，不包含阵列方向增益、极化或发射功率变化。对每个1.027584 s时间窗的斥距样本计算

$$Q_i=\left\langle R_i(t)^{-2}\right\rangle,\qquad
\Delta L_i=-10\log_{10}\left(\frac{Q_i}{Q_{\rm Event2}}\right).$$

In [ ]:
PATH_LOSS_OUTPUT_DIR = RESULTS_DIR / '5_flux_density' / 'path_loss'
PATH_LOSS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

mean_inverse_square_range = {}
for event_id, rows in direction_timeseries.groupby('event_id', sort=False):
    ranges_km = rows['slant_range_km'].to_numpy(float)
    mean_inverse_square_range[event_id] = float(np.mean(ranges_km ** -2))

reference_q = mean_inverse_square_range['Event 2']
path_loss_rows = []
for event_id in ['Missing 1', 'Missing 2']:
    path_loss_rows.append({
        'event_id': event_id,
        'path_attenuation_relative_to_event2_db': -10.0 * np.log10(
            mean_inverse_square_range[event_id] / reference_q
        ),
    })

relative_path_loss = pd.DataFrame(path_loss_rows)
relative_path_loss.to_csv(
    PATH_LOSS_OUTPUT_DIR / 'missing_events_relative_path_loss.csv',
    index=False, float_format='%.12e',
)
display(relative_path_loss)
print('Saved to:', PATH_LOSS_OUTPUT_DIR / 'missing_events_relative_path_loss.csv')
